In [1]:
from pathlib import Path
import os, sys

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if Path.cwd().resolve() != PROJECT_ROOT:
    os.chdir(PROJECT_ROOT)

root_str = str(PROJECT_ROOT)
if root_str in sys.path:
    sys.path.remove(root_str)
sys.path.insert(0, root_str)

#print("PROJECT_ROOT =", PROJECT_ROOT)
#print("cwd =", Path.cwd())

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import scanpy as sc
import pandas as pd
import numpy as np
import os
import torch

## preprocess and embedding

In [18]:
from src.preprocessing import pp
from sklearn.model_selection import train_test_split
import scvi

In [19]:
control_key = "is_control"
condition_keys = "guide_merged"
condition_rep_keys = "gene_embeddings"
random_seed = 42
dataset_name = "Norman"
sample_rep = "X_pca" 
#sample_rep = "X_scVI" 
#sample_rep = "X_flatvi"
#sample_rep = "X_state"

In [20]:
if sample_rep == "X_state":
    filePath = './data/raw/adata_Training_state_emb.h5ad'
    if not os.path.exists("./data/raw/adata_Training_state_emb.h5ad"):
        !state emb transform --model-folder ./data/SE-600M --input ./data/raw/adata_Training.h5ad --output ./data/raw/adata_Training_state_emb.h5ad
else:
    filePath = './data/raw/my_norman.h5ad'
adata = sc.read_h5ad(filePath)
#adata = adata[adata.obs.sample(frac=0.1, random_state=42).index].to_memory()
adata.layers["counts"] = adata.X.copy()
print(adata)

AnnData object with n_obs × n_vars = 101719 × 33694
    obs: 'guide_identity', 'UMI_count', 'gemgroup', 'number_of_cells', 'guide_merged'
    var: 'gene_symbols'
    layers: 'counts'


In [21]:
adata.obs[control_key] = (adata.obs[condition_keys] == "ctrl")
gene_list = adata[adata.obs[control_key]==False].obs[condition_keys].unique()
print(adata.obs[control_key].value_counts())

is_control
False    93324
True      8395
Name: count, dtype: int64


In [22]:
rng = np.random.default_rng(random_seed) 
test_ratio = 0.1
gene_list = list(gene_list)
zero_shot = True

if not zero_shot:
    # 分层抽样 先验证分布内学习能力
    adata_pert = adata[adata.obs[control_key] == False].copy()
    y = adata_pert.obs[condition_keys].astype(str).values
    idx = np.arange(adata_pert.n_obs)
    train_idx, test_idx = train_test_split(
        idx,
        test_size=test_ratio,
        random_state=random_seed,
        stratify=y 
    )
    adata_train = adata_pert[train_idx].copy()
    adata_test = adata_pert[test_idx].copy()
    adata_control = adata[adata.obs[control_key] == True].copy()
    print(gene_list)
    del adata, adata_pert
else:
    # 按基因分割 zero-shot
    n_test = max(1, int(len(gene_list) * test_ratio))
    test_gene = rng.choice(gene_list, size=n_test, replace=False).tolist()
    print(test_gene)
    train_gene = [g for g in gene_list if g not in test_gene]
    print(train_gene)
    adata_control = adata[adata.obs[control_key]==True].copy() # control的target_gene是non-targeting
    adata_train = adata[adata.obs[condition_keys].isin(train_gene)].copy() 
    adata_test = adata[adata.obs[condition_keys].isin(test_gene)].copy()
    del adata

['AHR+FEV', 'ZBTB10+SNAI1', 'FOXF1+ctrl', 'ctrl+CEBPA', 'DUSP9+IGDCC3', 'DUSP9+KLF1', 'ZC3HAV1+ctrl', 'ELMSAN1+ctrl', 'IGDCC3+ZBTB25', 'ZNF318+FOXL2', 'OSR2+ctrl', 'MAP2K6+IKZF3', 'ctrl+HOXB9', 'FEV+ctrl', 'UBASH3B+CNN1', 'EGR1+ctrl', 'HNF4A+ctrl', 'PTPN12+UBASH3A', 'CDKN1B+CDKN1A', 'ctrl+KLF1', 'CEBPB+OSR2', 'FOXL2+MEIS1', 'CBL+PTPN12', 'ctrl+UBASH3A', 'SGK1+TBX2', 'HES7+ctrl', 'JUN+CEBPB', 'SAMD1+TGFBR2']
['TSC22D1+ctrl', 'KLF1+MAP2K6', 'CEBPE+RUNX1T1', 'MAML2+ctrl', 'ctrl+CEBPE', 'CBL+PTPN9', 'LHX1+ELMSAN1', 'TGFBR2+ETS2', 'SGK1+TBX3', 'DUSP9+ctrl', 'MAP2K6+SPI1', 'ctrl+ELMSAN1', 'UBASH3B+ctrl', 'UBASH3B+PTPN12', 'ctrl+FOXA1', 'FOXA3+FOXA1', 'ETS2+IGDCC3', 'BCORL1+ctrl', 'MEIS1+ctrl', 'GLB1L2+ctrl', 'KLF1+ctrl', 'PTPN12+OSR2', 'BAK1+ctrl', 'MAP2K3+SLC38A2', 'CBL+ctrl', 'ctrl+ETS2', 'ctrl+FEV', 'ctrl+SET', 'TBX3+ctrl', 'LHX1+ctrl', 'KLF1+FOXA1', 'TBX3+TBX2', 'SLC4A1+ctrl', 'RREB1+ctrl', 'ZNF318+ctrl', 'DUSP9+MAPK1', 'COL2A1+ctrl', 'ctrl+ZBTB25', 'MAP4K5+ctrl', 'CEBPE+KLF1', 'SLC6A9+c

In [23]:
n_comps = 128
n_hidden = 2048
n_layers = 2
condition_rep_dict = pd.read_pickle("./data/processed/norman_gene_pert.pkl")
model_ref = None
model_train = None
model_test = None
scvi_save_path = f"./data/processed/model/{sample_rep}_ncomps{n_comps}_hidden{n_hidden}_layers{n_layers}_{dataset_name}_{random_seed}_{test_ratio}_{zero_shot}"
flatvi_save_path = f"./data/processed/model/{sample_rep}_ncomps{n_comps}_hidden{n_hidden}_layers{n_layers}_{dataset_name}"
state_decoder_save_path = f"./data/processed/model/{sample_rep}_{dataset_name}"
model_save_path = None

load_embedding_model = True
if sample_rep in ["X_scVI"]:
    model_save_path = scvi_save_path
    if load_embedding_model:
        try:
            model_ref = scvi.model.SCVI.load(f"{model_save_path}_ref", adata=adata_control)
        except (FileNotFoundError, OSError, ValueError) as e:
            model_ref = None 
        try:
            model_train = scvi.model.SCVI.load(f"{model_save_path}_train", adata=adata_train)
        except (FileNotFoundError, OSError, ValueError) as e:
            model_train = None
        if adata_test is not None:
            try:
                model_test = scvi.model.SCVI.load(f"{model_save_path}_test", adata=adata_test)
            except (FileNotFoundError, OSError, ValueError) as e:
                model_test = None
elif sample_rep == "X_flatvi":
    model_save_path = flatvi_save_path
    if load_embedding_model:
        try:
            model_ref = scvi.model.SCVI.load(f"{flatvi_save_path}_ref", adata=adata_control)
        except (FileNotFoundError, OSError, ValueError) as e:
            model_ref = None 
elif sample_rep=="X_state":
    from src.preprocessing import build_train_eval_loaders,NBDecoderTrainer,NBDecoder
    model_save_path = state_decoder_save_path
    z_dim = adata_control.obsm["X_state"].shape[1] # 2058
    n_genes = adata_control.n_vars
    decoder = NBDecoder(z_dim=z_dim, n_genes=n_genes, hidden=(1024,2048,4096), dropout=0.1)
    train_loader, val_loader = build_train_eval_loaders(
                                    adata_train=adata_control,
                                    adata_eval=adata_train,   
                                    count_layer="counts",
                                    emb_key="X_state",
                                    batch_size=256,
                                )
    trainer = NBDecoderTrainer(decoder, lr=1e-4, device="cuda", use_amp=True)
    trainer.fit(train_loader, val_loader=val_loader, epochs=50)
    trainer.save(f"{state_decoder_save_path}.pt")

In [24]:
adata_control, adata_train, adata_test, model_ref, model_train, model_test = pp.process_to_embedding( 
    adata_control,
    adata_train,
    adata_test = adata_test,
    sample_rep = sample_rep,
    n_comps = n_comps,
    n_hidden = n_hidden,
    n_layers = n_layers,
    model_ref = model_ref,
    model_train = model_train,
    model_test = model_test,
    model_save_path = model_save_path,
    control_key = control_key,
    condition_keys = condition_keys,
    condition_rep_keys = condition_rep_keys,
    condition_rep_dict = condition_rep_dict,
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    )
if sample_rep == "X_pca":
    sample_rep_scaled = sample_rep + "_scaled" # 额 别忘了
else:
    sample_rep_scaled = sample_rep

[0.9999997  0.9999994  1.0000012  0.99999917 0.9999999  0.9999996
 1.0000012  0.99999833 1.0000008  0.99999785 1.0000004  0.9999998
 0.99999803 0.99999905 0.99999994 1.0000004  1.0000001  0.999999
 0.999999   1.0000015  1.0000001  1.0000001  1.0000008  1.0000007
 0.99999934 1.         0.9999989  0.99999845 1.0000005  1.0000023
 0.9999989  1.0000001  0.9999994  1.0000012  1.0000013  0.9999996
 0.99999845 1.000001   1.0000007  1.0000008  1.         1.0000015
 0.99999946 1.0000004  0.9999991  0.9999996  1.0000008  0.99999887
 0.9999998  1.0000014  1.0000017  1.0000012  0.9999991  1.000001
 0.9999995  0.9999993  1.000001   1.0000001  1.0000008  1.0000007
 0.99999726 0.9999993  0.99999803 1.0000008  1.0000023  1.0000006
 0.99999917 0.99999934 1.0000005  1.0000005  1.0000005  1.0000002
 0.9999998  0.9999994  1.0000012  1.0000018  0.9999988  0.99999857
 1.0000006  1.0000001  1.0000019  1.0000015  0.9999998  0.9999996
 0.9999981  1.0000012  0.9999994  0.99999917 0.9999995  0.9999996
 1.0000006

In [25]:
preprocess_save_path = f"./data/processed/{dataset_name}_{random_seed}_{test_ratio}_{zero_shot}_{sample_rep_scaled}_{n_comps}"
adata_control.write_h5ad(f"{preprocess_save_path}_control.h5ad")
adata_train.write_h5ad(f"{preprocess_save_path}_train.h5ad")
if adata_test is not None:
    adata_test.write_h5ad(f"{preprocess_save_path}_test.h5ad")

In [26]:
print(adata_control)
print(adata_train)
print(adata_test)

AnnData object with n_obs × n_vars = 8395 × 33694
    obs: 'guide_identity', 'UMI_count', 'gemgroup', 'number_of_cells', 'guide_merged', 'is_control'
    var: 'gene_symbols'
    uns: 'log1p', 'pca'
    obsm: 'X_pca', 'X_pca_scaled'
    varm: 'X_mean', 'PCs'
    layers: 'counts', 'X_centered'
AnnData object with n_obs × n_vars = 85487 × 33694
    obs: 'guide_identity', 'UMI_count', 'gemgroup', 'number_of_cells', 'guide_merged', 'is_control'
    var: 'gene_symbols'
    uns: 'log1p'
    obsm: 'X_pca', 'X_pca_scaled', 'gene_embeddings'
    layers: 'counts'
AnnData object with n_obs × n_vars = 7837 × 33694
    obs: 'guide_identity', 'UMI_count', 'gemgroup', 'number_of_cells', 'guide_merged', 'is_control'
    var: 'gene_symbols'
    uns: 'log1p'
    obsm: 'X_pca', 'X_pca_scaled', 'gene_embeddings'
    layers: 'counts'


In [ ]:
denoised_df = model_ref.get_normalized_expression(adata_control, return_mean=True,library_size=1e4)
raw_adata = adata_control.copy()
sc.pp.normalize_total(raw_adata, target_sum=1e4)
raw_matrix = raw_adata.X
if hasattr(raw_matrix, "toarray"):
    raw_matrix = raw_matrix.toarray()

In [ ]:
# 计算原始数据和重建数据的基因均值
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
mean_raw = np.mean(raw_matrix, axis=0)
mean_recon = np.mean(denoised_df.values, axis=0)

# 计算相关性 (Pearson 或 Spearman)
corr, _ = pearsonr(mean_raw, mean_recon)
print(f"Gene Mean Correlation (Raw vs Recon): {corr:.4f}")

# 可视化
plt.figure(figsize=(6, 6))
plt.scatter(mean_raw, mean_recon, s=1, alpha=0.5)
plt.plot([0, max(mean_raw)], [0, max(mean_raw)], 'r--') # 对角线
plt.xlabel("Raw Mean Expression (Normalized)")
plt.ylabel("Reconstructed Mean Expression")
plt.title(f"Reconstruction Quality (R = {corr:.2f})")
plt.xscale('log')
plt.yscale('log')
plt.show()